In [ ]:
# Cell 1: Environment Installation
!pip install xgboost pandas scikit-learn fastapi uvicorn mlflow PyYAML httpx pydantic-settings pytest pytest-asyncio

# Cell 2: Trigger the Pipe-and-Filter Training Phase
from src.train import run_pipe_and_filter_training
run_pipe_and_filter_training()

# Cell 3: Boot ALL 3 Independent Microservice Node Clusters in Parallel
import subprocess
import time

print("Launching Microservice Node 1: Fraud Prediction Cluster [Port 8001]...")
f_proc = subprocess.Popen(["python", "-m", "src.agents.fraud_predictor"])
time.sleep(2)

print("Launching Microservice Node 2: Pipe-and-Filter Compliance RAG [Port 8002]項目...")
r_proc = subprocess.Popen(["python", "-m", "src.agents.compliance_rag"])
time.sleep(2)

print("Launching Microservice Node 3: Central Saga & CQRS Gateway [Port 8000]...")
g_proc = subprocess.Popen(["python", "-m", "src.main"])

print("\n🎉 Everything is fully mapped and running! Send payloads to: http://127.0.0.1:8000/api/v1/adjudicate")

In [ ]:
# Run this for testing
import httpx
import json

url = "http://127.0.0.1:8000/api/v1/adjudicate"

# TEST 1: The Happy Path (Low Risk Claim)
payload_happy = {
    "policy_holder_id": "POL_BITS_001",
    "customer_age": 34,
    "policy_deductible": 1000,
    "claim_amount": 45000.0,
    "past_claims_count": 0,
    "incident_hour": 14
}

# TEST 2: The Early Exit Path (High-Risk Fraud Profile)
payload_fraud = {
    "policy_holder_id": "POL_BITS_002",
    "customer_age": 21,
    "policy_deductible": 500,
    "claim_amount": 140000.0,
    "past_claims_count": 4,
    "incident_hour": 2  # Deep night + high amount triggers early exit
}

with httpx.Client() as client:
    print("--- SENDING HAPPY PATH PAYLOAD ---")
    r1 = client.post(url, json=payload_happy)
    print(json.dumps(r1.json(), indent=2))
    
    print("\n--- SENDING ANOMALOUS FRAUD PAYLOAD ---")
    r2 = client.post(url, json=payload_fraud)
    print(json.dumps(r2.json(), indent=2))